# 05. Map research keywords to Reddit topics

The PubMed and ClinicalTrials.gov pulls used clinical search terms. The Reddit posts are tagged with our own topic names. This notebook links the two so a topic means the same thing across all three sources, then loads the result into the database.

**Run from `notebooks/Reddit_Data/`.**

**Input:** `data/interim/submissions_topic_analytics_long_form.parquet` (from notebook 04)
**Output:** `data/interim/matched_topics.parquet`, plus the `staging_reddit_posts` table

Database credentials come from a `.env` file at the project root. See `.env.example` for the variable names. Do not commit `.env`.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DBUSER')}:{os.getenv('DBPASSWORD')}"
    f"@{os.getenv('DBHOST')}:{os.getenv('DBPORT')}/{os.getenv('DBNAME')}"
)

In [ ]:
long_df = pd.read_parquet("../../data/interim/submissions_topic_analytics_long_form.parquet")
long_df.info()

## Mapping dictionary

Keys are the keywords used to query PubMed and ClinicalTrials.gov. Values are the topic names assigned to Reddit posts in notebook 04. Several keywords can point at the same topic, which is the whole reason this step exists.

In [ ]:
# keyword: what is in the database for pubmed and clinical trials
# topic: reddit topics from topic analysis
KEYWORD_TO_TOPIC = {
    "menstrual cycle": "menstrual_cycle",
    "irregular menstruation": "irregular_periods",
    "menorrhagia": "heavy_bleeding",
    "heavy menstrual bleeding": "heavy_bleeding",
    "abnormal uterine bleeding": "abnormal_bleeding",
    "dysmenorrhea": "dysmenorrhea",
    "period cramps": "dysmenorrhea",
    "amenorrhea": "amenorrhea",
    "breakthrough bleeding": "abnormal_bleeding",
    "intermenstrual bleeding": "abnormal_bleeding",
    "combined oral contraceptives": "oral_contraceptives",
    "progestin-only pill": "oral_contraceptives",
    "IUD": "iud",
    "intrauterine device": "iud",
    "copper IUD": "iud",
    "levonorgestrel IUD": "iud",
    "contraceptive side effects": "contraceptive_side_effects",
    "birth control and mood": "contraceptive_side_effects",
    "birth control and libido": "contraceptive_side_effects",
    "tubal ligation": "tubal_ligation",
    "vulvovaginal candidiasis": "yeast_infection",
    "recurrent yeast infection": "yeast_infection",
    "bacterial vaginosis": "bacterial_vaginosis",
    "urinary tract infection": "uti",
    "UTI": "uti",
    "boric acid vaginal suppository": "boric_acid",
    "vaginal pH": "vaginal_ph_microbiome",
    "vaginal microbiome": "vaginal_ph_microbiome",
    "probiotics vaginal health": "probiotics",
    "pelvic pain": "pelvic_pain",
    "ovarian cyst": "ovarian_cyst",
    "ovarian torsion": "ovarian_torsion",
    "endometriosis": "endometriosis",
    "adenomyosis": "adenomyosis",
    "pelvic floor dysfunction": "pelvic_floor",
    "pelvic floor physical therapy": "pelvic_floor",
    "vulvodynia": "vulvodynia",
    "interstitial cystitis": "interstitial_cystitis",
    "Bartholin cyst": "bartholin_cyst",
    "breast lump": "breast_lump",
    "fibroadenoma": "fibroadenoma",
    "breast pain": "breast_pain",
    "mastalgia": "breast_pain",
    "breast cancer screening": "breast_cancer_screening",
    "mammography": "breast_cancer_screening",
    "breast ultrasound": "breast_cancer_screening",
    "nipple pain": "breast_pain",
    "PCOS": "pcos",
    "polycystic ovary syndrome": "pcos",
    "hormonal acne": "hormonal_acne",
    "hirsutism": "hirsutism",
    "thyroid dysfunction": "thyroid",
    "hypothyroidism": "thyroid",
    "Hashimoto's thyroiditis": "thyroid",
    "perimenopause": "menopause",
    "menopause hormone therapy": "menopause",
    "HRT": "menopause",
    "hot flashes": "hot_flashes_night_sweats",
    "night sweats": "hot_flashes_night_sweats",
    "sexually transmitted infection": "sti_std",
    "STI": "sti_std",
    "painful intercourse": "painful_sex",
    "dyspareunia": "painful_sex",
    "libido": "libido",
    "sexual desire": "libido",
    "unprotected sex": "unprotected_sex",
    "pregnancy test": "pregnancy_test",
    "medical abortion": "abortion",
    "misoprostol": "abortion",
    "mifepristone": "abortion",
    "abortion access": "abortion_policy",
    "hair loss": "hair_loss",
    "alopecia": "hair_loss",
    "iron deficiency anemia": "iron_anemia",
    "ferritin": "iron_anemia",
    "bloating": "bloating",
    "hemorrhoids": "hemorrhoids",
    "nausea": "nausea",
    "fatigue": "fatigue_sleep",
    "sleep disturbance": "fatigue_sleep",
    "heart palpitations": "heart_palpitations",
    "headache": "headache_migraine",
    "migraine": "headache_migraine",
    "allergic": "allergic_reaction",
}

## Pairing function

Adds a `keyword` column based on the topic column. A topic with several keywords will produce several rows, so we drop exact duplicates on post plus topic plus keyword afterward.

Any topic with no matching keyword gets flagged rather than silently dropped. That is how we caught the topics that came out of BERTopic after the API pull had already run, which never had a keyword to begin with.

In [ ]:
def build_topic_to_keywords(keyword_to_topic: dict) -> dict:
    # Invert keyword -> topic into topic -> list of keywords
    topic_to_keywords = {}
    for keyword, topic in keyword_to_topic.items():
        topic_to_keywords.setdefault(topic, []).append(keyword)
    return topic_to_keywords


def add_keyword_column(
    df: pd.DataFrame,
    keyword_to_topic: dict,
    topic_col: str = "matched_topic",
    id_cols: list = None,
) -> pd.DataFrame:
    topic_to_keywords = build_topic_to_keywords(keyword_to_topic)

    # Lookup table with one row per topic-keyword pair
    lookup_df = pd.DataFrame([
        {topic_col: topic, "keyword": keyword}
        for topic, keywords in topic_to_keywords.items()
        for keyword in keywords
    ])

    # Rows explode where a topic maps to several keywords
    merged = df.merge(lookup_df, on=topic_col, how="left")

    unmatched = merged[merged["keyword"].isna()][topic_col].unique()
    if len(unmatched) > 0:
        print(f"Warning: no keyword found for {len(unmatched)} topic(s): {list(unmatched)}")

    dedup_subset = id_cols + [topic_col, "keyword"] if id_cols else None
    before = len(merged)
    merged = merged.drop_duplicates(subset=dedup_subset)
    print(f"Dropped {before - len(merged)} duplicate row(s).")

    return merged.reset_index(drop=True)

## Apply it

In [ ]:
long_df_matched = add_keyword_column(
    long_df,
    keyword_to_topic=KEYWORD_TO_TOPIC,
    topic_col="matched_topics",
    id_cols=["created_utc"],
)
long_df_matched.info()

In [ ]:
long_df_matched.to_parquet("../../data/interim/matched_topics.parquet")

## Load into the database

Only the identifier, topic, and keyword go into the staging table. Post text stays out of the database entirely.

In [ ]:
matched_table = long_df_matched[["created_utc", "matched_topics", "keyword"]]

matched_table.to_sql(
    name="staging_reddit_posts",
    con=engine,
    index=False,
    if_exists="replace",
)
print(f"Uploaded {len(matched_table)} rows to staging_reddit_posts")